In [1]:
import tensorflow as tf
from tensorflow import keras
from keras import layers , Sequential

## Load Dataset

In [3]:
from keras.datasets import imdb

In [ ]:
VOCAB_SIZE = 1000

(X_train , y_train) , (X_test , y_test) = imdb.load_data(num_words = VOCAB_SIZE)

In [10]:
print(f"Training Samples : {X_train.shape}")
print(f"Test Samples : {X_test.shape}")

Training Samples : (25000,)
Test Samples : (25000,)


In [24]:
print(X_train[0][:20])

print(f"lenght of first sample (word) : {len(X_train[0])}")

print(f"Label of First Sample (word) : {y_train[0]}")


[1, 14, 22, 16, 43, 530, 973, 2, 2, 65, 458, 2, 66, 2, 4, 173, 36, 256, 5, 25]
lenght of first sample (word) : 218
Label of First Sample (word) : 1


## padding the Sequences

In [32]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LEN = 200 # her review ko 200 word ki fised len pe le aayegnge 
    # - agar kisi review me 200 se kam word hue to zero se pad karenge
    # - agar kisi review me 200 se jada word hue to cut kar denge to make it 200 words

X_train_padded = pad_sequences(X_train , maxlen = MAX_LEN)
X_test_padded = pad_sequences(X_test , maxlen = MAX_LEN)

print(f"""X_train_padded : {X_train_padded.shape},
X_test_padded : {X_test_padded.shape}
""")

X_train_padded : (25000, 200),
X_test_padded : (25000, 200)



In [37]:
model = Sequential([

    layers.Input(shape=(MAX_LEN,)),

    layers.Embedding(
        input_dim = VOCAB_SIZE , output_dim = 128
    ),
    layers.SimpleRNN(
        64 , activation = "tanh"
    ),
    layers.Dense(
        1 , activation = "sigmoid"
    )
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 200, 128)       │       128,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 140,417 (548.50 KB)

 Trainable params: 140,417 (548.50 KB)

 Non-trainable params: 0 (0.00 B)

In [39]:
model.compile(
    optimizer = "adam",
    loss = "binary_crossentropy",
    metrics = ["accuracy"]
)

In [41]:
history = model.fit(
    X_train_padded , y_train,
    epochs = 5,
    batch_size = 64,
    validation_data = (X_test_padded , y_test)
)

Epoch 1/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - accuracy: 0.7230 - loss: 0.5391 - val_accuracy: 0.7020 - val_loss: 0.5916
Epoch 2/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - accuracy: 0.7752 - loss: 0.4855 - val_accuracy: 0.7142 - val_loss: 0.5931
Epoch 3/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.7610 - loss: 0.4979 - val_accuracy: 0.7090 - val_loss: 0.5913
Epoch 4/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - accuracy: 0.8051 - loss: 0.4360 - val_accuracy: 0.7523 - val_loss: 0.5490
Epoch 5/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.8171 - loss: 0.4122 - val_accuracy: 0.7413 - val_loss: 0.5592


In [42]:
# Test set pe accuracy
test_loss, test_acc = model.evaluate(X_test_padded, y_test)
print(f"Test Accuracy: {test_acc:.4f}")

# Apna khud ka sentence test 
word_index = imdb.get_word_index()

def predict_sentiment(text):
    words = text.lower().split()
    encoded = [word_index.get(word, 0) + 3 for word in words]  # +3 kyunki IMDB me offset hai
    padded = pad_sequences([encoded], maxlen=MAX_LEN)
    prediction = model.predict(padded)[0][0]
    return "Positive" if prediction > 0.5 else "Negative", prediction

result, score = predict_sentiment("this movie was absolutely amazing and wonderful")
print(f"Sentiment: {result} (score: {score:.4f})")

782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7413 - loss: 0.5592
Test Accuracy: 0.7413
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
Sentiment: Positive (score: 0.8303)
